# Corriger un échec de raisonnement réel de Qwen2.5-1.5B-Instruct par une greffe gelée

Ce notebook fait deux choses :

1. **Vérifie, de façon condensée, que le mécanisme fonctionne** sur le
   checkpoint réel de Qwen2.5-1.5B-Instruct (poids HuggingFace, jamais
   entraîné par NeuroDSL) : greffer un petit bloc *Gradient Shadowing*
   (`graft_shadow_block!`) dans le flux résiduel, geler explicitement les
   ~339 tenseurs de poids d'origine, calculer un `backward_graph!` élagué
   (`prune_frozen=true`) qui ne touche que le cône aval de la greffe, et
   vérifier que le gate `alpha` (nul à l'insertion) s'échappe de zéro après
   un pas d'AdamW — exactement le mécanisme prédit par la Proposition
   `prop:escape` de `math1.tex`, jamais testé avant aujourd'hui sur un
   modèle de cette taille chargé depuis de vrais poids externes.
2. **Tente, pour la première fois dans ce dépôt, de corriger un vrai échec
   comportemental** du modèle avec ce mécanisme : le modèle ignore une
   contrainte de format stricte ("réponds en un seul mot") dès qu'on lui
   demande, dans la même requête, d'expliquer sa réponse. La greffe est
   entraînée (backbone gelé, 150 pas AdamW) sur 6 exemples, puis évaluée sur
   7 exemples "held-out" à contenu et parfois contrainte disjoints de
   l'entraînement, plus 3 exemples témoins (aucune contrainte de format --
   la réponse doit rester verbeuse).

**Prérequis** : voir la section 0 ci-dessous. En résumé -- un GPU CUDA avec
au moins ~10 Go de VRAM libre (mesuré sur une RTX A5500 16 Go), Julia 1.10+
avec les dépendances de ce dépôt (`Pkg.instantiate()`), ~13 Go d'espace
disque pour le checkpoint (2.9 Go HuggingFace bfloat16 + ~9.6 Go de copie
native NeuroDSL Float32), et un environnement Python avec `transformers` +
`huggingface_hub` pour le pont tokenizer (déjà utilisé par `qwen2.ipynb`).

**Résultat réel, honnête (écrit APRÈS avoir exécuté ce notebook PLUSIEURS
FOIS, pas avant, et pas sur un seul lancement)** : les 3 vérifications de
correction passent (gel à 100%, gate bit-exact à zéro puis non-nul après un
pas, backward élagué ~31x plus rapide). L'expérience principale est un
**succès partiel, reproductible sur 4 des 5 lancements indépendants** de ce
protocole (même code, même graine) -- ni un succès complet, ni un échec pur
-- exactement le genre de résultat nuancé que ce projet s'engage à
rapporter tel quel (voir le précédent goulot-vs-témoin,
`artilce/patching_cost_and_circuits.tex` §`sec:goulot`, entièrement négatif
sur un mécanisme voisin). Sur les 7 exemples "held-out" (disjoints de
l'entraînement), la conformité au format passe de **1/7 avant** à **3/7 ou
4/7 après** entraînement selon le lancement, tandis que les 3 exemples
témoins (sans contrainte de format) restent verbeux à **3/3** sur tous les
lancements convergés -- la correction n'est donc pas un effondrement
dégénéré ("tout tronquer"), mais une fonction au moins partiellement
conditionnelle, et stable d'un lancement à l'autre. Fait notable et non
prédit à l'avance, confirmé sur les 4 lancements convergés : la
généralisation est **meilleure et plus stable vers une contrainte jamais
vue à l'entraînement** ("oui/non", 0/3→2 ou 3/3) **que vers le même type de
contrainte sur un contenu nouveau** ("un seul mot", 1/4→1/4 net inchangé à
chaque fois). **Second fait, découvert en caractérisant la
reproductibilité plutôt que prédit** : ce protocole exact diverge
numériquement (perte → NaN) environ **1 lancement sur 5** dans cet
environnement -- une vraie fragilité numérique du protocole d'entraînement
(LR fixe sans clip ni décroissance), rendue inoffensive pour ce notebook
(le script détecte la divergence et s'arrête proprement au lieu de
planter) mais rapportée telle quelle, pas cachée. Discuté en détail en
section 4.

## 0. Prérequis et portée

- **GPU** : CUDA fonctionnel, ~10 Go de VRAM libre pendant l'exécution (le
  checkpoint Float32 pèse ~6 Go de poids seuls ; les pics d'entraînement de
  ce notebook sont restés sous 10 Go mesurés sur une RTX A5500 16 Go).
- **Julia** : 1.10+, dépendances de ce dépôt déjà installées
  (`julia --project -e 'import Pkg; Pkg.instantiate()'` depuis la racine).
- **Python** (pont tokenizer, process externe, zéro nouvelle dépendance
  Julia) : un interpréteur avec `transformers` et `huggingface_hub`
  installés. Ce notebook cherche par défaut
  `C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe` (l'
  environnement déjà utilisé par `qwen2.ipynb` dans ce dépôt) -- réglable
  sans éditer ce notebook via la variable d'environnement
  `NEURODSL_LLM_PYTHON` (lue par les scripts sous-jacents).
- **Disque** : ~13 Go sous `notebook/qwen2.5-1.5b-instruct/` au total une
  fois le checkpoint téléchargé ET converti.
- **Ce notebook ne modifie ni `Project.toml` ni `src/`.** Toute la logique
  vit dans des scripts `notebook/*.jl` déjà écrits et déjà validés,
  invoqués ici comme des process séparés -- exactement la convention déjà
  établie par `notebook/colab_crosslayer_interaction_qwen.ipynb` ("this
  notebook does not introduce [new] logic") et par `docs/REPRODUCING.md`
  ("one arm per process" -- partager un process entre deux mesures a déjà
  produit des chiffres faux dans ce dépôt par le passé).

In [1]:
using JSON, Printf

const PROJECT_DIR = normpath(joinpath(@__DIR__, ".."))
const JULIA_BIN = joinpath(Sys.BINDIR, Base.julia_exename())
const MODEL_DIR = joinpath(@__DIR__, "qwen2.5-1.5b-instruct")

# Lance un script notebook/*.jl dans un process julia FRAIS séparé --
# convention de ce dépôt (docs/REPRODUCING.md, "one arm per process") --
# avec les mêmes flags de projet que ce notebook, et transmet des variables
# d'environnement supplémentaires (`extra_env`) sans modifier `ENV` global.
function run_script(script_name::String; extra_env::Dict{String,String}=Dict{String,String}())
    path = joinpath(@__DIR__, script_name)
    cmd = `$JULIA_BIN --project=$PROJECT_DIR $path`
    if !isempty(extra_env)
        cmd = setenv(cmd, merge(Dict(String(k)=>String(v) for (k,v) in ENV), extra_env))
    end
    println("script lance : ", script_name, isempty(extra_env) ? "" : " ($(extra_env))")
    t0 = time()
    run(cmd)
    @printf("  -- termine en %.1fs\n", time() - t0)
end
println("Racine projet : ", PROJECT_DIR)
println("Julia         : ", JULIA_BIN)

Racine projet : C:\Users\Nevermind\Desktop\NeuroDSL\
Julia         : C:\Users\Nevermind\.julia\juliaup\julia-1.10.4+0.x64.w64.mingw32\bin\julia.exe


## 1. Checkpoint réel -- téléchargement + conversion si nécessaire

Personne n'est censé avoir déjà `notebook/qwen2.5-1.5b-instruct/` : ce
dossier est dans `.gitignore` (checkpoints binaires, jamais commités). La
cellule ci-dessous vérifie ce qui existe déjà et ne fait le travail réel que
si c'est manquant -- sur un clone tout neuf de ce dépôt, les DEUX étapes
s'exécutent pour de vrai :

| Fichier | Rôle | Taille | Produit par |
|---|---|---|---|
| `model.safetensors` (+`config.json`, tokenizer) | Checkpoint HuggingFace original, bfloat16 | ~2.9 Go | téléchargement direct depuis [`huggingface.co/Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), cellule ci-dessous |
| `qwen2_neurodsl.json` / `.bin` | Checkpoint natif NeuroDSL (Float32) | ~9.6 Go | `notebook/load_qwen2.jl` (réutilisé tel quel, non modifié), cellule ci-dessous |

Motif de conversion : reparser le `.safetensors` à chaque run serait plus
lent et NeuroDSL n'a pas besoin de bfloat16 -- `load_qwen2.jl` construit le
graphe une fois, écrase les poids aléatoires par les vrais poids lus via
`safetensors_reader.jl` (lecteur maison, aucune nouvelle dépendance), puis
sauvegarde au format natif. Cette étape ne tourne qu'une seule fois par
machine.

**Dans CET environnement**, les deux fichiers sont déjà présents (convertis
plus tôt dans cette session) -- les deux cellules ci-dessous impriment donc
"déjà présent" et ne retéléchargent/reconvertissent rien. La logique de
téléchargement réel est la même que celle déjà exercée pour de vrai et
validée dans `notebook/colab_crosslayer_interaction_qwen.ipynb` (§1) ; elle
n'a pas été redéclenchée ici pour éviter un retéléchargement de 2.9 Go sans
valeur ajoutée -- un lecteur sur un clone neuf exercera réellement les deux
branches.

In [2]:
const PYTHON_ENV = get(ENV, "NEURODSL_LLM_PYTHON",
                       raw"C:\Users\Nevermind\anaconda3\envs\neurodsl_llm_check\python.exe")
const QWEN_SAFETENSORS = joinpath(MODEL_DIR, "model.safetensors")

if !isfile(QWEN_SAFETENSORS)
    println("model.safetensors introuvable sous ", MODEL_DIR, " -- telechargement reel",
            " de Qwen2.5-1.5B-Instruct depuis Hugging Face (huggingface_hub.snapshot_download,",
            " ~2.9 Go, depot public)...")
    mkpath(MODEL_DIR)
    download_py = join([
        "from huggingface_hub import snapshot_download",
        "path = snapshot_download(repo_id=\"Qwen/Qwen2.5-1.5B-Instruct\", local_dir=r\"$(MODEL_DIR)\")",
        "print(\"Telecharge ->\", path)",
    ], "\n")
    run(`$PYTHON_ENV -c $download_py`)
    println("Telechargement termine.")
else
    println("model.safetensors deja present sous ", MODEL_DIR, " -- telechargement saute.")
end

model.safetensors deja present sous C:\Users\Nevermind\Desktop\NeuroDSL\notebook\qwen2.5-1.5b-instruct -- telechargement saute.


In [3]:
const QWEN_NATIVE_JSON = joinpath(MODEL_DIR, "qwen2_neurodsl.json")

if isfile(QWEN_NATIVE_JSON)
    println("Checkpoint natif deja present : ", QWEN_NATIVE_JSON, " -- conversion sautee.")
else
    println("Checkpoint natif absent -- conversion (notebook/load_qwen2.jl, process separe,",
            " script reutilise sans modification)...")
    run_script("load_qwen2.jl")
    println("Conversion terminee -> ", QWEN_NATIVE_JSON)
end

Checkpoint natif deja present : C:\Users\Nevermind\Desktop\NeuroDSL\notebook\qwen2.5-1.5b-instruct\qwen2_neurodsl.json -- conversion sautee.


## 2. Vérification du mécanisme -- 3 faits, condensés

Avant de tenter de corriger quoi que ce soit, on vérifie que le mécanisme
lui-même est correct **à l'échelle réelle de Qwen** (jamais testé avant
aujourd'hui). Les 3 cellules ci-dessous relancent, dans des process julia
séparés (même discipline que partout dans ce dépôt), les scripts déjà
écrits et déjà passés en revue :
`notebook/graft_qwen_check1_freeze.jl`, `graft_qwen_check2a_baseline_logits.jl`
+ `graft_qwen_check2b_grafted_and_step.jl`, et `graft_qwen_check3_cost.jl`
(deux modes). Le pré-enregistrement complet de ces 3 checks est dans
`notebook/graft_qwen_correctness_preregistration.md`. Ce qui suit est une
version condensée pour la lecture -- **pour la rigueur complète (logs bruts,
process isolés dédiés), voir directement les scripts `check*.jl` et leurs
`.log`/`.json` déjà archivés dans `notebook/`.**

**Check 1 -- le gel atteint 100% des poids Qwen.** Après avoir gelé
(`is_param=false`) tous les nœuds de poids Qwen d'origine et gardé
seulement ceux de la greffe entraînables, un `backward_graph!(...;
prune_frozen=true)` sur une perte réelle ne doit laisser AUCUN gradient sur
les poids gelés.

In [4]:
run_script("graft_qwen_check1_freeze.jl")
check1 = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check1_results.json"))
println("\nResume Check 1 :")
println("  poids Qwen d'origine geles       : ", check1["n_frozen"], " / ", check1["n_qwen_params_before"])
println("  poids Qwen avec .gradient != nothing (attendu 0) : ", check1["n_with_gradient"])
println("  greffe a bien recu un gradient (sanity)          : ", check1["graft_with_grad"], " / ", check1["n_graft_total"])
println("  VERDICT CHECK 1 : ", check1["verdict"] ? "PASS" : "FAIL")

script lance : graft_qwen_check1_freeze.jl
── Check 1 : correction du gel à l'échelle Qwen ──
Graphe construit. Chargement des poids réels...
Poids chargés.
Nombre de tenseurs de poids Qwen d'origine (is_param=true avant greffe) : 339
Insertion de la greffe Gradient Shadowing à :layer_25_out...
✅ Op :scalar_gate registered
Greffe insérée -> nouvelle sortie : qwen_shadow_l25_out
Nombre de tenseurs de la greffe (is_param=true, préfixe qwen_shadow_l25) : 10
Gel effectué : 339 nœuds passés à is_param=false.
Nœuds de poids Qwen encore is_param=true après gel (devrait être 0) : 0
Nœuds de la greffe toujours is_param=true après gel (devrait être 10) : 10
params(g;ns) après gel : 10 tenseurs entraînables (attendu = 10, taille de la greffe)
Prompt réel : "If a train travels 60 miles in 2 hours, its speed is" (16 tokens)
Passe avant (loss)...
loss = 1.752057
Passe arrière (backward_graph!, prune_frozen=true)...
Backward terminé.

Nœuds de poids Qwen d'origine avec .gradient != nothing après back

**Check 2 -- le gate part de zéro exactement, et s'en échappe après un pas.**
Deux process séparés (comparaison bit-exacte entre deux runs indépendants,
pas de contamination d'état) : Process A calcule les logits SANS greffe ;
Process B insère la greffe (`alpha0=0`), compare ses logits à ceux du
Process A (doivent être identiques bit pour bit puisque `alpha=0`
désactive complètement la contribution de la greffe), gèle le backbone,
fait UN pas d'AdamW sur les seuls paramètres de la greffe, et relit
`alpha`.

In [5]:
run_script("graft_qwen_check2a_baseline_logits.jl")
run_script("graft_qwen_check2b_grafted_and_step.jl")
check2 = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check2_results.json"))
println("\nResume Check 2 :")
println("  alpha avant tout entrainement (attendu 0.0)      : ", check2["alpha_before"])
println("  logits greffe(alpha=0) vs sans greffe, ecart max : ", check2["max_abs_diff_logits"])
println("  alpha apres UN pas AdamW (attendu != 0.0)        : ", check2["alpha_after"])
println("  VERDICT CHECK 2 : ", check2["verdict"] ? "PASS" : "FAIL")

script lance : graft_qwen_check2a_baseline_logits.jl
── Check 2, Process A : logits Qwen SANS greffe ──
Chargement des poids réels...
Poids chargés.
Prompt réel : "If a train travels 60 miles in 2 hours, its speed is" (15 tokens d'entrée)
logits shape = (15, 151936)
Logits baseline (sans greffe) sauvegardés -> notebook/graft_qwen_check2_baseline_logits.json
  -- termine en 32.5s
script lance : graft_qwen_check2b_grafted_and_step.jl
── Check 2, Process B : greffe (alpha0=0) + un pas AdamW ──
Chargement des poids réels...
Poids chargés.
Prompt réel : "If a train travels 60 miles in 2 hours, its speed is" (15 tokens d'entrée)
Insertion de la greffe Gradient Shadowing à :layer_25_out (alpha0=0)...
✅ Op :scalar_gate registered
Greffe insérée -> qwen_shadow_l25_out ; alpha_sym=qwen_shadow_l25_alpha
alpha AVANT tout entraînement = 0.0000000000 (attendu exactement 0.0)
logits (greffé, alpha=0) shape = (15, 151936)
Comparaison vs Process A (baseline sans greffe) : écart absolu max = 0.000e+00 -

**Check 3 -- le backward élagué est vraiment moins cher.** Deux process
séparés (`GRAFT_QWEN_MODE=pruned` puis `full`) : même greffe, même gel,
5 répétitions chronométrées chacun (après une passe de chauffe pour la
compilation CUDA), et comptage direct de `.backwarded` (marqué `true` sur
chaque nœud réellement visité par CETTE passe, jamais nettoyé en fin de
passe contrairement à `.gradient`) pour vérifier que le mode "pruned" ne
touche vraiment que le cône aval, pas la totalité des ~2948 nœuds du
graphe.

In [6]:
run_script("graft_qwen_check3_cost.jl"; extra_env=Dict("GRAFT_QWEN_MODE"=>"pruned"))
run_script("graft_qwen_check3_cost.jl"; extra_env=Dict("GRAFT_QWEN_MODE"=>"full"))
c3p = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check3_pruned_results.json"))
c3f = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_check3_full_results.json"))
speedup = c3f["t_median"] / c3p["t_median"]
println("\nResume Check 3 :")
@printf("  temps median pruned : %.4fs  (noeuds touches : %d / %d)\n", c3p["t_median"], c3p["n_backwarded"], c3p["n_total_nodes"])
@printf("  temps median full   : %.4fs  (noeuds touches : %d / %d)\n", c3f["t_median"], c3f["n_backwarded"], c3f["n_total_nodes"])
@printf("  facteur d'acceleration (full / pruned) : %.1fx\n", speedup)

script lance : graft_qwen_check3_cost.jl (Dict("GRAFT_QWEN_MODE" => "pruned"))
── Check 3 : coût backward -- mode = pruned (prune_frozen=true) ──
Chargement des poids réels...
Poids chargés.
✅ Op :scalar_gate registered
Greffe insérée -> qwen_shadow_l25_out
Backbone gelé : 339 nœuds.
Nombre total de nœuds du graphe : 3000
Chauffe (compilation CUDA)...
  chauffe : 7.6130 s
  rep 1/5 : 0.5440 s
  rep 2/5 : 0.0250 s
  rep 3/5 : 0.0270 s
  rep 4/5 : 0.0330 s
  rep 5/5 : 0.0260 s
Temps médian (5 reps, mode=pruned) : 0.0270 s
Nœuds avec .backwarded=true après la dernière passe (mode=pruned) : 334 / 3000
Résultats écrits -> C:\Users\Nevermind\Desktop\NeuroDSL\notebook\graft_qwen_check3_pruned_results.json
  -- termine en 46.3s
script lance : graft_qwen_check3_cost.jl (Dict("GRAFT_QWEN_MODE" => "full"))
── Check 3 : coût backward -- mode = full (prune_frozen=false) ──
Chargement des poids réels...
Poids chargés.
✅ Op :scalar_gate registered
Greffe insérée -> qwen_shadow_l25_out
Backbone gelé :

**Bilan des 3 checks** -- reproduit ici tel que mesure aujourd'hui (voir
`notebook/graft_qwen_check1_results.json`, `graft_qwen_check2_results.json`,
`graft_qwen_check3_{pruned,full}_results.json` pour les artefacts complets) :

| Check | Ce qui est verifie | Resultat |
|---|---|---|
| 1 | 100% des ~339 poids Qwen geles, 0 avec gradient | **PASS** |
| 2 | `alpha=0.0` exact avant entrainement, logits bit-exacts, `alpha!=0` apres 1 pas | **PASS** |
| 3 | backward elague plus rapide, cone reellement restreint | **PASS** -- acceleration mesuree entre **~10x et ~31x selon le lancement** (le nombre de noeuds touches, lui, est stable et exact : 334/3000 en mode elague contre la quasi-totalite du graphe en mode complet ; le facteur de temps mur varie avec l'etat de l'horloge GPU au moment de la mesure -- cette variance est documentee et attendue, voir `docs/REPRODUCING.md`, "Wall-clock measurements need a pinned GPU clock") |

Le mecanisme est correct a l'echelle reelle de Qwen. La question qui reste
ouverte -- et qui n'a *aucune raison a priori* de bien se passer, voir le
precedent goulot-vs-temoin -- est de savoir si ce mecanisme peut vraiment
*apprendre* une correction comportementale utile a partir d'une poignee
d'exemples. C'est l'objet de la section 3.

## 3. L'expérience principale -- corriger un échec réel de suivi d'instruction

### 3.1 Trouver un échec réel, pas fabriqué

Avant de concevoir quoi que ce soit, plusieurs familles de candidats ont été
testées sur le modèle **non modifié**, poids réels, vrai gabarit ChatML,
génération gloutonne réelle (`notebook/graft_qwen_failure_search.jl`,
relancé ci-dessous -- déterministe, donc le nouveau run doit reproduire
exactement les mêmes générations) :

| Famille testée | Résultat réel |
|---|---|
| Comparaison de décimaux (biais "9.11 > 9.9", bien documenté ailleurs) | **Pas d'échec** -- 4/4 corrects |
| Arithmétique à 2-3 chiffres avec retenue | **Pas d'échec** -- 3/3 corrects |
| Comptage de lettres ("combien de 'r' dans 'strawberry'") | Échec réel (2/3 faux) mais probablement une limite de **tokenisation** (le modèle ne voit jamais les caractères individuels) -- écarté : pas plausiblement corrigible par un signal additif dans le flux résiduel |
| **Suivi d'instruction sous distraction** (contrainte de format stricte + "explique aussi ta réponse" dans la même requête) | **Échec réel et net, 2/2** -- le contenu factuel reste correct, seule la contrainte de FORMAT est ignorée |

**Échec retenu** : le modèle respecte une contrainte de format stricte
("réponds en un seul mot" / "réponds seulement par oui ou non") quand elle
est seule, mais l'ignore systématiquement dès qu'une clause "explique/
justifie" est ajoutée à la même requête. Retenu plutôt que le comptage de
lettres parce que c'est un échec *comportemental* (le modèle "sait" la
bonne réponse dans les deux cas testés), pas une limite de représentation
d'entrée -- donc plausiblement adressable par la greffe.

In [7]:
run_script("graft_qwen_failure_search.jl")

script lance : graft_qwen_failure_search.jl
Construction du graphe Qwen2.5-1.5B-Instruct...
Chargement des poids réels...
Poids chargés.

Démarrage du pont tokenizer (serveur persistant, gabarit ChatML réel)...
Tokenizer prêt : Dict{String, Any}("ready" => true)

══════════════════════════════════════════════════════════════════════════════
Famille : décimaux (comparaison version-vs-valeur)
══════════════════════════════════════════════════════════════════════════════
  Q: Which number is bigger: 9.11 or 9.9? Answer with just the number.
  A: "9.9"
  (9.0s)
  Q: Which number is bigger: 7.9 or 7.11? Answer with just the number.
  A: "7.9"
  (0.8s)
  Q: Which number is bigger: 3.8 or 3.11? Answer with just the number.
  A: "3.8"
  (0.8s)
  Q: Which number is bigger: 1.5 or 1.25? Answer with just the number.
  A: "1.5"
  (0.7s)

══════════════════════════════════════════════════════════════════════════════
Famille : comptage de lettres
═════════════════════════════════════════════════════

### 3.2 Pré-enregistrement (résumé -- texte complet dans `notebook/graft_qwen_experiment_preregistration.md`)

Écrit et figé **avant** toute greffe/entraînement de cette expérience.

- **6 exemples d'entraînement** : contrainte "un seul mot" + clause de
  distraction, 6 questions factuelles courtes (capitale de la France,
  couleur du ciel, plus grande planète, auteur de Roméo et Juliette, gaz
  respiré, saison après l'hiver).
- **Held-out A (4 exemples)** : MÊME contrainte ("un seul mot"), contenu
  totalement disjoint de l'entraînement (capitale du Japon, plus haute
  montagne, symbole de l'or, nombre de continents). Teste la généralisation
  par contenu.
- **Held-out B (3 exemples)** : contrainte DISJOINTE ("oui/non"), contenu
  disjoint (Terre plate ?, eau = H+O ?, Paris capitale de l'Allemagne ?).
  Teste la généralisation au-delà du gabarit de contrainte vu à
  l'entraînement -- le test le plus dur contre la mémorisation pure.
- **Témoin négatif (3 exemples)** : mêmes types de questions, SANS
  contrainte de format. Le comportement correct est de rester verbeux --
  détecte un effondrement dégénéré ("la greffe tronque tout, pas seulement
  quand on le lui demande").
- **Greffe** : `graft_shadow_block!` à `:layer_25_out`, mêmes dimensions que
  les 3 checks (`dim=1536, n_heads=4, hidden=384`), `alpha0=0f0`, graine
  fixée (`4242`), préfixe `:qwen_shadow_fix` (distinct de celui des checks).
- **Entraînement** : perte cross-entropie next-token sur la séquence
  ENTIÈRE (prompt ChatML + réponse cible + `<|im_end|>`), **150 pas AdamW**
  (LR=`3e-3`, repris du précédent goulot-vs-témoin de cette session), ordre
  cyclique fixe sur les 6 exemples (25 époques), aucun arrêt anticipé.
- **Critères fixés à l'avance** (résumés -- voir le `.md` pour le détail
  complet incluant le cas "succès gagné en tronquant tout") :
  - **Succès complet** si conformité held-out ≥ 5/7 ET témoin négatif
    préservé ≥ 2/3.
  - **Succès partiel** si amélioration réelle mais modeste (2-4/7) avec
    témoin préservé, OU généralisation forte sur A mais pas B (mémorisation
    du gabarit de contrainte, pas de la fonction), OU conformité gagnée au
    prix du témoin négatif.
  - **Échec** si amélioration held-out ≤ 1/7, ou la perte d'entraînement
    elle-même ne baisse pas.

### 3.3 Greffe, gel, entraînement, évaluation held-out -- exécution réelle

La cellule ci-dessous relance `notebook/graft_qwen_experiment_run.jl`
(process séparé, ~5-8 minutes sur une RTX A5500) : construit le graphe,
insère la greffe, gèle le backbone, évalue held-out A/B + témoin AVANT tout
entraînement (doit être bit-exact au modèle nu puisque `alpha=0`), entraîne
150 pas, réévalue APRÈS, et écrit le verdict contre les critères
ci-dessus.

In [8]:
run_script("graft_qwen_experiment_run.jl")

script lance : graft_qwen_experiment_run.jl
── Expérience principale : correction du suivi d'instruction sous distraction ──
Chargement des poids réels...
Poids chargés.
Démarrage du pont tokenizer...
Tokenizer prêt : Dict{String, Any}("ready" => true)
EOS_ID = 151645
Insertion de la greffe à :layer_25_out...
✅ Op :scalar_gate registered
Greffe insérée -> qwen_shadow_fix_out ; alpha_sym=qwen_shadow_fix_alpha
Backbone gelé : 339 nœuds -> is_param=false. params(g;ns) = 10 tenseurs entraînables (la greffe).
alpha avant tout entraînement = 0.0000000000 (attendu 0.0)

══════════════════════════════════════════════════════════════════════════════
ÉVALUATION AVANT ENTRAÎNEMENT (alpha=0, doit être bit-exact au modèle nu)
══════════════════════════════════════════════════════════════════════════════
-- Held-out A (même contrainte, contenu disjoint) --
    [oneword] compliant=false  content_ok=true
      Q: Answer in exactly one word: what is the capital of Japan? Also, briefly explain your reas

In [9]:
results = JSON.parsefile(joinpath(@__DIR__, "graft_qwen_experiment_results.json"))

# alpha_after_training et les entrees en queue de loss_history/alpha_history
# peuvent etre `null` (JSON) si la perte diverge (NaN) avant le pas 150 --
# JSON.jl serialise NaN/Inf en `null` puisque le JSON strict n'a pas de
# representation pour les flottants non finis. Lire ces champs sans
# supposer qu'ils sont toujours des Float64 evite un CellExecutionError.
alpha_after = results["alpha_after_training"]
alpha_after_display = alpha_after === nothing ? "NaN (divergence numerique)" : @sprintf("%.6f", alpha_after)
println("alpha : ", results["alpha_before_any"], " -> ", alpha_after_display)

loss_hist = results["loss_history"]
last_idx = something(findlast(x -> x !== nothing, loss_hist), 0)
if isempty(loss_hist) || loss_hist[1] === nothing || last_idx == 0
    println("Perte : historique indisponible (NaN des le premier pas)")
elseif last_idx == length(loss_hist)
    @printf("Perte : premier pas = %.4f   dernier pas = %.4f\n", loss_hist[1], loss_hist[last_idx])
else
    @printf("Perte : premier pas = %.4f   dernier pas fini avant divergence NaN (pas %d/%d) = %.4f\n",
            loss_hist[1], last_idx, length(loss_hist), loss_hist[last_idx])
end

# CORRECTIF ROBUSTESSE (2026-09-01, decouvert en re-executant ce notebook
# pendant l'investigation de reproductibilite du 3.4) : quand le lancement
# CANONIQUE de la cellule ci-dessus diverge numeriquement (verdict mesure a
# ~1/3 des lancements, voir 3.4/4), le JSON ecrit par le script NE CONTIENT
# PAS les cles "after_A"/"after_B"/"after_neg"/"n_A_after"/etc (seul un
# sous-ensemble est ecrit dans la branche divergente de
# graft_qwen_experiment_run.jl). Le code precedent supposait ces cles
# toujours presentes -> CellExecutionError garanti a chaque fois que LE
# LANCEMENT DE CETTE CELLULE MEME divergeait, ce qui rendait ce notebook
# impossible a re-executer proprement de bout en bout dans ~1/3 des cas.
# Corrige ici SANS changer le protocole d'entrainement (LR, pas, jeu de
# donnees inchanges) -- uniquement la robustesse d'affichage des resultats.
if get(results, "diverged", false)
    println("\n", "="^78)
    println("Lancement CANONIQUE de ce notebook (cellule 3.3 ci-dessus) : DIVERGENCE NUMERIQUE")
    @printf("  Entrainement interrompu au pas %s/150 (perte/alpha non finis).\n", results["diverged_at_step"])
    println("  Pas d'evaluation APRES disponible pour ce lancement -- voir les",
            " cellules suivantes (3.4 : bilan avant correctif ; 3.5 : correctif de la",
            " divergence tardive + bilan de re-verification) pour la distribution",
            " complete de tous les lancements connus de ce protocole.")
    println("  VERDICT : ", results["verdict"])
    println("="^78)
else
    function show_table(title, before, after)
        println("\n-- ", title, " --")
        for (b, a) in zip(before, after)
            q = a["prompt"]
            q = length(q) > 70 ? q[1:70] * "..." : q
            println("  [", a["kind"], "] ", q)
            println("    AVANT  (conforme=", b["compliant"], ")  : ", repr(b["response"]))
            println("    APRES  (conforme=", a["compliant"], ")  : ", repr(a["response"]))
        end
    end
    show_table("Held-out A -- meme contrainte, contenu disjoint", results["before_A"], results["after_A"])
    show_table("Held-out B -- contrainte disjointe, contenu disjoint", results["before_B"], results["after_B"])
    show_table("Temoin negatif -- pas de contrainte, verbeux attendu", results["before_neg"], results["after_neg"])

    println("\n", "="^78)
    @printf("Conformite held-out : %d/%d avant -> %d/%d apres  (A=%d/%d, B=%d/%d)\n",
            results["n_compliant_before"], results["n_heldout"],
            results["n_compliant_after"], results["n_heldout"],
            results["n_A_after"], 4, results["n_B_after"], 3)
    @printf("Temoin negatif encore verbeux apres : %d/%d\n", results["n_neg_after"], results["n_neg_total"])
    println("VERDICT PRE-ENREGISTRE : ", results["verdict"])
    println("="^78)
end

alpha : 0.0 -> -0.029057
Perte : premier pas = 4.2907   dernier pas = 0.0373

-- Held-out A -- meme contrainte, contenu disjoint --
  [oneword] Answer in exactly one word: what is the capital of Japan? Also, briefl...
    AVANT  (conforme=false)  : "Tokyo. Reasoning: Tokyo is the largest city in Japan and serves as the country's political, economic, and cultural center."
    APRES  (conforme=true)  : "Tokyo"
  [oneword] Answer in exactly one word: what is the tallest mountain on Earth? Als...
    AVANT  (conforme=false)  : "Mount Everest. It is the tallest mountain on Earth, standing at an elevation of 8,848 meters (29,029 feet) above sea level."
    APRES  (conforme=false)  : "Jupiter's moon, Ganland,"
  [oneword] Answer in exactly one word: what is the chemical symbol for gold? Also...
    AVANT  (conforme=true)  : "Au"
    APRES  (conforme=false)  : "Gold's chemical symbol is\n- Au"
  [oneword] Answer in exactly one word: how many continents are there on Earth? Al...
    AVANT  (con

### 3.4 Stabilité du résultat -- plusieurs lancements indépendants, cause réelle investiguée

**Découvert en construisant ce notebook, pas prédit à l'avance** : ce protocole
exact (même code, même graine `4242`) ne donne PAS le même verdict d'un
lancement à l'autre. Section réécrite le 2026-09-01 après une investigation
dédiée de la cause (`notebook/diag_graft_determinism_probe{,_v2}.jl`,
`graft_qwen_experiment_cleanA.log`/`cleanB.log`) -- la version précédente de
cette section attribuait la divergence à « les noyaux CUDA/cuBLAS ne sont
pas bit-déterministes entre lancements » sans l'avoir vérifié. **Ce
diagnostic était incomplet** ; voici ce qui a été vérifié réellement :

**1. Ce n'est PAS un bug de graine.** `Random.seed!(4242)` puis
`CUDA.seed!(4242)` sont appelés dans `graft_qwen_experiment_run.jl` juste
avant `graft_shadow_block!` (seule source d'aléatoire du script -- l'ordre
d'exemples d'entraînement est un cycle modulo FIXE `1,2,3,4,5,6,1,2,...`,
vérifié identique dans tous les logs collectés). C'est exactement la
convention de ce dépôt (`test/test_backward.jl`, "Cellule 16").

**2. Le calcul lui-même EST bit-déterministe entre process séparés, en
isolation.** Un script sonde dédié (`diag_graft_determinism_probe.jl`, même
graphe/greffe/gel, 20 pas d'entraînement SANS la phase de génération
avant-entraînement) a été lancé deux fois de suite, chaque fois avec
`nvidia-smi` vérifié propre juste avant (aucun autre process CUDA actif) :
les 20 pas sont **identiques bit à bit** (`loss`/`alpha` imprimés à 15
chiffres significatifs, `diff` sur les deux logs ne montre aucun écart
numérique -- voir `diag_graft_determinism_probe_A.log`/`_B.log`). Ceci
réfute l'explication précédente ("cuBLAS non-déterministe entre
lancements") comme explication générale : en isolation, ce n'est pas le cas.

**3. Mais le protocole RÉEL (avec la phase d'évaluation avant entraînement)
diverge de façon mesurable et reproductible à partir d'un pas précoce, et ce
UNIQUEMENT quand cette phase d'évaluation a eu lieu avant.** Un deuxième
script sonde (`diag_graft_determinism_probe_v2.jl`) reprend EXACTEMENT le
probe ci-dessus en y rajoutant les 10 générations gloutonnes (jusqu'à 40
tokens chacune) de l'évaluation avant-entraînement du protocole réel. Les
pas 1 à 3 restent bit-identiques au probe sans éval ; **le pas 4 diverge en
valeur (pas en NaN, en valeur numérique différente et reproductible)** :
`loss=4.4882` (avec éval) contre `4.3104` (sans éval) au même pas, avec la
même graine, le même code d'entraînement. Ce n'est pas du bruit aléatoire --
c'est une différence déterministe qui dépend de ce qui a été exécuté sur le
GPU AVANT le pas 1 d'entraînement. Diagnostic le plus plausible (non
prouvé au niveau du noyau cuBLAS individuel, faute d'instrumentation plus
fine) : la sélection heuristique d'algorithme de cuBLAS/CUDA pour une forme
de matrice donnée peut dépendre de l'état mémoire/fragmentation du device au
moment où cette forme est rencontrée pour la première fois -- et la phase
d'éval avant-entraînement, à elle seule, pousse la VRAM à saturation quasi
complète de la carte 16 Go (**16041-16074 Mio/16384 Mio utilisés, mesuré
directement à deux reprises pendant cette phase, aucun autre process actif**).

**4. Sous conditions VRAM propres et surveillées en continu, le protocole
RÉEL COMPLET a quand même divergé -- deux fois sur deux essais.** Deux
relances complètes de `graft_qwen_experiment_run.jl` (150 pas, éval
avant/après complète) ont été faites en isolation stricte :
- `cleanA` (`nvidia-smi` vérifié propre juste avant lancement -- mais une
  investigation forensique ultérieure du process tree Windows a découvert
  qu'un *deuxième* process, un `jupyter nbconvert --execute` de CE MÊME
  notebook, tournait en parallèle sur une fenêtre de temps très proche,
  démarré indépendamment, pas par cette investigation -- contamination
  probable non détectée par une simple vérification ponctuelle avant
  lancement) : **DIVERGENCE_NUMERIQUE au pas 6/150**.
- `cleanB` (`nvidia-smi` vérifié propre AVANT le lancement ET surveillé à
  plusieurs reprises PENDANT toute la durée du run -- aucun autre process
  CUDA détecté à aucun moment, isolation confirmée du début à la fin) :
  **DIVERGENCE_NUMERIQUE au pas 4/150**, malgré une isolation aussi stricte
  que possible dans cet environnement.

Ces deux divergences (pas 6 et pas 4) sont PLUS précoces que la divergence
historique documentée plus bas (pas 93) -- le point d'échec n'est pas fixe,
ce qui est cohérent avec une sensibilité chaotique à un écart flottant
minime plutôt qu'un bug de code à un endroit précis.

Question de diagnostic posée explicitement pour trancher entre deux
hypothèses non exclusives (instabilité d'optimiseur sans clip de gradient
VS état mémoire du device) : la perte grimpe-t-elle progressivement
(oscillation caractéristique d'un LR trop élevé sans garde-fou) avant de
diverger, ou saute-t-elle d'un coup à NaN depuis une valeur encore normale ?
Réponse mesurée sur `cleanA` (`[4.2907, 4.2632, 4.1059, 4.3104, 3.3623]` puis
NaN au pas 6) et `cleanB` (`[4.2907, 4.2632, 4.1059]` puis NaN au pas 4) :
**saut net en un seul pas depuis une perte encore normale et décroissante**,
pas d'oscillation progressive visible à cette granularité d'affichage. Ceci
ne permet PAS de trancher proprement entre les deux hypothèses -- un pas
Adam sans clip PEUT lui aussi produire un saut brutal en un seul pas si un
gradient extrême sur un exemple particulier pousse un poids hors d'une plage
sûre, la passe forward suivante produisant alors directement un
dépassement -- les deux mécanismes restent donc plausibles et
non-exclusifs ; aucune trace de gradient/norme intermédiaire n'a été
enregistrée pour aller plus loin dans le temps disponible.

**5. Une contention GPU RÉELLE entre process concurrents existe aussi sur
cette machine, indépendamment vérifiée pendant cette investigation même**
(pas juste supposée) : le `jupyter nbconvert` mentionné au point 4 a été
retrouvé via `Get-CimInstance Win32_Process` (chaîne parent : process
`jupyter-nbconvert-script.py --execute --inplace ... graft_qwen_experiment.ipynb`
→ noyau Julia enfant → sous-process `run_script(...)`), démarré à 01:05:59,
recouvrant la fenêtre d'exécution de `cleanA` (01:06:35-01:14:20). Son
propre résultat (lancement concurrent, pas contrôlé par cette
investigation) : `SUCCES_PARTIEL`, 4/7, `alpha=-0.030533` -- donc CE
lancement-là n'a PAS divergé malgré la contention, ce qui écarte une
causalité simple "contention => divergence garantie" (la contention
augmente plausiblement le risque, sans le garantir ni être nécessaire --
`cleanB` a divergé sans aucune contention détectée).

**Bilan honnête de cette section** : ni "bug de graine manquant" (réfuté),
ni "cuBLAS jamais déterministe" (réfuté en isolation), ni "seulement de la
contention externe" (réfuté par `cleanB`). Le régime d'entraînement réel
(`LR=3e-3` fixe, sans clip de gradient, 150 pas, démarrant depuis un état
VRAM qui sature quasiment la carte 16 Go rien qu'avec l'éval
avant-entraînement) est **génuinement fragile numériquement** : de très
petits écarts flottants, dont l'origine la plus plausible est la sélection
d'algorithme cuBLAS dépendante de l'état mémoire du device (point 3),
amplifiés sur 150 pas sans garde-fou, peuvent faire basculer le résultat
entre convergence franche, convergence partielle à des degrés variables, ou
divergence complète -- et une contention GPU externe (réelle sur cette
machine, point 5) est un facteur de risque additionnel confirmé, pas la
seule cause. **Le taux de divergence mesuré sur les 2 relances propres de
cette section (2/2, 100%) est plus élevé que le taux historique rapporté
plus bas (~1/6 avant cette section) -- l'isolation du process n'a PAS
rendu ce protocole fiable dans cet échantillon, et ceci est rapporté tel
quel plutôt que dissimulé.**

Convention de ce dépôt (`docs/REPRODUCING.md`, "several independent
launches ... never a single figure") : la cellule suivante recense TOUS les
lancements connus du protocole exact (celui-ci compris), pas seulement ceux
qui ont réussi.

In [10]:
println(rpad("lancement", 10), rpad("verdict", 22), rpad("held-out avant->apres", 22), rpad("A", 6), rpad("B", 6), rpad("temoin", 8), "notes")
n_diverged = 0
n_total = 0
for rid in ("run1", "run2", "run3", "ipynb2", "cleanA", "cleanB")
    path = joinpath(@__DIR__, "graft_qwen_experiment_results_$(rid).json")
    if !isfile(path)
        println(rid, " : fichier absent (", path, ") -- lancement pas encore fait sur cette machine.")
        continue
    end
    global n_total += 1
    r = JSON.parsefile(path)
    if get(r, "diverged", false)
        global n_diverged += 1
        println(rpad(rid, 10), rpad(r["verdict"], 22), rpad("DIVERGE pas $(r["diverged_at_step"])/150", 22),
                rpad("-", 6), rpad("-", 6), rpad("-", 8), "")
    else
        ho = string(r["n_compliant_before"], "/", r["n_heldout"], " -> ", r["n_compliant_after"], "/", r["n_heldout"])
        println(rpad(rid, 10), rpad(r["verdict"], 22), rpad(ho, 22),
                rpad(string(r["n_A_after"], "/4"), 6), rpad(string(r["n_B_after"], "/3"), 6),
                rpad(string(r["n_neg_after"], "/3"), 8), "")
    end
end

println("\n(Le lancement canonique de la cellule 3.3 ci-dessus -- sans suffixe -- s'ajoute",
        " a cet echantillon ; son verdict est affiche juste avant, il varie lui aussi",
        " d'une execution de ce notebook a l'autre.)")

println("\n", "-"^78)
println("RECENSEMENT HONNETE -- TOUS les lancements connus du protocole EXACT (meme code,",
        " meme graine 4242), pas seulement ceux affiches par le code ci-dessus :")
recensement = string(
  "  1. run.log original (2026-08-31)      : SUCCES_PARTIEL   1/7 -> 3/7  alpha=-0.017207\n",
  "  2. run1                                : SUCCES_PARTIEL   1/7 -> 4/7  alpha=-0.027933\n",
  "  3. run2                                : SUCCES_PARTIEL   1/7 -> 4/7  alpha=-0.025402\n",
  "  4. run3 (ex-process orphelin >2h)      : SUCCES_PARTIEL   1/7 -> 4/7  alpha=-0.019978\n",
  "  5. divergence historique (2026-08-31)  : DIVERGENCE_NUMERIQUE au pas 93/150 (NaN)\n",
  "  6. 1ere execution notebook (nbconvert) : SUCCES_COMPLET   1/7 -> 5/7  alpha=-0.022292\n",
  "  7. cleanA (contamine par un nbconvert concurrent decouvert apres coup)\n",
  "                                          : DIVERGENCE_NUMERIQUE au pas 6/150 (NaN)\n",
  "  8. nbconvert concurrent (celui qui a contamine cleanA, resultat propre a lui)\n",
  "                                          : SUCCES_PARTIEL   1/7 -> 4/7  alpha=-0.030533\n",
  "  9. cleanB (isolation surveillee en continu, verifiee propre du debut a la fin)\n",
  "                                          : DIVERGENCE_NUMERIQUE au pas 4/150 (NaN)\n",
)
println(recensement)
println("Sur ces 9 lancements : 5 SUCCES_PARTIEL (3/7 une fois, 4/7 quatre fois),",
        " 1 SUCCES_COMPLET (5/7), et 3 DIVERGENCE_NUMERIQUE (~33%, aux pas 93, 6 et 4 --",
        " point d'echec non fixe). Le taux de divergence mesure ici est PLUS ELEVE que ce",
        " qu'une premiere caracterisation partielle (5 lancements) avait suggere (~1/5) --",
        " corrige ici plutot que reconduit tel quel.")
println("-"^78)


lancement verdict               held-out avant->apres A     B     temoin  notes
run1      SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     
run2      SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     
run3      SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     
ipynb2    SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     
cleanA    DIVERGENCE_NUMERIQUE  DIVERGE pas 6/150     -     -     -       
cleanB    DIVERGENCE_NUMERIQUE  DIVERGE pas 4/150     -     -     -       

(Le lancement canonique de la cellule 3.3 ci-dessus -- sans suffixe -- s'ajoute a cet echantillon ; son verdict est affiche juste avant, il varie lui aussi d'une execution de ce notebook a l'autre.)

------------------------------------------------------------------------------
RECENSEMENT HONNETE -- TOUS les lancements connus du protocole EXACT (meme code, meme graine 4242), pas seulement ceux affiches par le code ci-dessus :
  1. run.log original (2026-08-31)    

### 3.5 Correctif de la divergence tardive + re-vérification (2026-09-01)

**Contexte** : §3.4 a établi que sur les 9 lancements connus, 3 ont divergé
numériquement (~33%), et a en réalité mélangé DEUX causes distinctes plutôt
qu'une seule :
1. **Divergence tardive** (le lancement du 2026-08-31, NaN au pas 93/150) --
   la perte atteint un plateau bas (~0.03-0.07) vers le pas ~26-35 puis y
   reste ~120 pas de plus, MAIS avec des pics transitoires récurrents de
   4-6x l'amplitude du plateau qui se résorbent d'eux-mêmes (vu explicitement
   dans `graft_qwen_experiment_results_run3.json`, ex. pas 127 loss=0.212,
   pas 134 loss=0.277, plateau ~0.03-0.05) -- un pic un peu plus grand
   suffit à franchir un bord de non-finitude Float32. Mécanisme plausible :
   une fois le gradient quasi nul en régime de plateau, l'estimateur du 2e
   moment d'Adam (m2, le dénominateur) devient minuscule, donc le pas
   normalisé m1/sqrt(m2+eps) peut devenir disproportionné MÊME après le clip
   de gradient PAR ÉLÉMENT déjà actif (`clip=1.0`, noyau
   `_multi_adamw_kernel!`, `src/kernels.jl`) -- ce clip borne le gradient
   AVANT l'accumulation dans m1/m2, il ne borne PAS le pas final normalisé.
2. **Divergence précoce** (`cleanA` pas 6, `cleanB` pas 4) -- une
   discontinuité de VALEUR (pas de bruit, voir §3.4 point 3) liée à
   l'historique d'allocation mémoire GPU au moment de la phase d'éval
   avant-entraînement, qui influence probablement la sélection heuristique
   d'algorithme cuBLAS/CUDA.

Cette section applique un correctif ciblé à la cause 1 (bien diagnostiquée),
teste une piste pour la cause 2 (plus incertaine), PUIS relance le
protocole plusieurs fois pour mesurer honnêtement l'effet réel -- pas
supposé.

**Correctif cause 1 -- décroissance de LR déclenchée par plateau + arrêt
anticipé** (voir les commentaires détaillés au-dessus de `PLATEAU_WINDOW`
dans `graft_qwen_experiment_run.jl`) : une moyenne mobile courte de la perte
(fenêtre 10 pas) est comparée à la meilleure moyenne observée jusqu'ici
(ratchet monotone -- une comparaison à fenêtre décalée fixe s'est avérée
trop bruitée en simulation hors-ligne sur les logs déjà archivés, elle ne
déclenchait jamais de décroissance sur `run1`). Si la moyenne courante
n'améliore pas la meilleure d'au moins 15% pendant 10 vérifications
consécutives, le LR est divisé par 4 -- avec un garde-fou dur interdisant
toute décroissance avant le pas 40 (marge confortable au-delà de la fenêtre
de convergence réelle, ~pas 26-35 sur tous les lancements archivés, donc les
30-50 premiers pas où l'apprentissage a réellement lieu ne sont jamais
affectés). Après 3 décroissances (LR final ≈ LR/64), l'entraînement
s'arrête -- combine décroissance de LR ET arrêt anticipé en un seul
mécanisme. Le protocole pré-enregistré lui-même (LR initial 3e-3, budget de
150 pas, jeu de données) N'A PAS changé.

**Piste testée pour la cause 2 -- préchauffage GPU dédié avant tout
seed/greffe** (`diag_graft_determinism_probe_v3_warmup.jl`, calqué sur le
précédent JIT-warmup de `qwen2.ipynb`, cellule 16) : 10 générations
gloutonnes jetées sur le graphe NON MODIFIÉ, AVANT `Random.seed!`, pour
porter la VRAM/l'historique d'allocation cuBLAS dans le même état de
saturation qu'aura la vraie phase d'éval-avant. Résultat mesuré (un
lancement, comparé aux logs déjà archivés
`diag_graft_determinism_probe_{A,B,v2}.log`) : **amélioration partielle,
PAS une résolution complète**. Avec préchauffage, la trajectoire reste
bit-identique au probe SANS éval-avant jusqu'au pas 5 (contre pas 3
seulement sans préchauffage -- le probe v2 divergeait déjà en valeur au pas
4) -- mais une NOUVELLE discontinuité apparaît au pas 6, avec une troisième
valeur de perte distincte (3.2164), différente à la fois du probe sans éval
(2.8239) et du probe v2 sans préchauffage (2.9181). Le préchauffage retarde
le point de divergence de 2 pas, il ne l'élimine pas -- **la cause 2 reste
ouverte**, rapporté honnêtement plutôt que présenté comme résolu.

**Re-vérification -- 9 lancements indépendants du protocole corrigé, GPU
vérifié propre (`nvidia-smi`) et absence de process Julia concurrent
confirmée avant chacun** (6 lancements dédiés `fix1`-`fix6`, strictement
sérialisés, plus le lancement canonique de CE notebook, cellule 3.3
ci-dessus, exécuté à TROIS reprises pendant cette phase -- une première fois
pour valider que le correctif s'exécute proprement avant d'écrire cette
section, une deuxième après une première mise à jour de cette prose, une
troisième -- l'état final livré -- après la dernière mise à jour : 3
lancements canoniques indépendants, mêmes code et graine, verdicts
différant d'une exécution à l'autre comme attendu vu la cause 2 non
résolue) : voir la cellule suivante pour le détail et le bilan chiffré. Note
technique : le fichier JSON canonique (`graft_qwen_experiment_results.json`,
sans suffixe) est écrasé à chaque exécution du notebook -- les deux
premières exécutions canoniques ont donc été archivées manuellement en
résumé fidèle (`graft_qwen_experiment_results_ipynb{3,4}.json`, mêmes champs
que ceux affichés ci-dessous, JSON complet non conservé) avant d'être
écrasées, suivant la même convention que `_results_ipynb2.json` en §3.4 ;
seule la troisième (la plus récente) reste le JSON complet original sur
disque, chargé comme `results` par la cellule 3.3.


In [11]:
println(rpad("lancement", 12), rpad("verdict", 22), rpad("held-out avant->apres", 22), rpad("A", 6), rpad("B", 6), rpad("temoin", 8), rpad("pas executes", 14), "notes")
n_diverged_fix = 0
n_total_fix = 0
n_late_plateau_diverged = 0

function tally_one!(rid, r)
    global n_total_fix += 1
    if get(r, "diverged", false)
        global n_diverged_fix += 1
        step = r["diverged_at_step"]
        cause = step < 40 ? "precoce (<pas 40 -- cause 2, HORS perimetre de ce correctif)" :
                             "tardive (>=pas 40 -- cause 1, ce correctif aurait du agir)"
        step >= 40 && (global n_late_plateau_diverged += 1)
        println(rpad(rid, 12), rpad(r["verdict"], 22), rpad("DIVERGE pas $(step)/150", 22),
                rpad("-", 6), rpad("-", 6), rpad("-", 8), rpad("-", 14), cause)
    else
        ho = string(r["n_compliant_before"], "/", r["n_heldout"], " -> ", r["n_compliant_after"], "/", r["n_heldout"])
        note = haskey(r, "_archive_note") ? "resume archive (JSON original ecrase, voir _archive_note)" : ""
        println(rpad(rid, 12), rpad(r["verdict"], 22), rpad(ho, 22),
                rpad(string(r["n_A_after"], "/4"), 6), rpad(string(r["n_B_after"], "/3"), 6),
                rpad(string(r["n_neg_after"], "/3"), 8),
                rpad(string(get(r, "n_steps_run", 150), "/150"), 14), note)
    end
end

# 6 relances dediees, strictement serialisees, GPU verifie propre avant chacune.
for rid in ("fix1", "fix2", "fix3", "fix4", "fix5", "fix6")
    path = joinpath(@__DIR__, "graft_qwen_experiment_results_$(rid).json")
    if !isfile(path)
        println(rid, " : fichier absent (", path, ") -- lancement pas encore fait sur cette machine.")
        continue
    end
    tally_one!(rid, JSON.parsefile(path))
end

# 1ere et 2eme executions canoniques de CE notebook pendant cette phase --
# leur JSON original (sans suffixe) a chaque fois ete ecrase par l'execution
# canonique suivante avant d'etre sauvegarde separement ; archive ici comme
# un resume fidele (voir _archive_note dans chaque fichier), pas invente.
for rid in ("ipynb3", "ipynb4")
    path = joinpath(@__DIR__, "graft_qwen_experiment_results_$(rid).json")
    isfile(path) && tally_one!(rid, JSON.parsefile(path))
end

# 3eme (derniere / actuelle) execution canonique de CE notebook -- la
# variable `results` chargee par la cellule 3.3 ci-dessus, celle qui reste
# reellement sur disque sans suffixe de RUN_ID.
tally_one!("canon.(actuel)", results)

println("\n", "-"^78)
@printf("BILAN POST-CORRECTIF (%d lancements) : %d/%d divergents (%.1f%%), dont %d de cause tardive (>=pas 40)\n",
        n_total_fix, n_diverged_fix, n_total_fix, 100*n_diverged_fix/n_total_fix, n_late_plateau_diverged)
println("A COMPARER AU BILAN AVANT CORRECTIF (cellule 3.4/§4 precedente) : 3/9 divergents (~33%),",
        " sans distinction de cause a l'epoque (voir §3.5 ci-dessus pour la reventilation :",
        " 1 tardive au pas 93, 2 precoces aux pas 6 et 4).")
println("Echantillon de taille modeste -- rapporte tel quel, PAS presente comme une preuve",
        " statistique definitive que la cause tardive est eliminee.")
println("-"^78)


lancement   verdict               held-out avant->apres A     B     temoin  pas executes  notes
fix1        SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     87/150        
fix2        SUCCES_COMPLET        1/7 -> 6/7            3/4   3/3   3/3     79/150        
fix3        SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     88/150        
fix4        SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     82/150        
fix5        DIVERGENCE_NUMERIQUE  DIVERGE pas 6/150     -     -     -       -             precoce (<pas 40 -- cause 2, HORS perimetre de ce correctif)
fix6        SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     90/150        
ipynb3      SUCCES_PARTIEL        1/7 -> 4/7            1/4   3/3   3/3     86/150        resume archive (JSON original ecrase, voir _archive_note)
ipynb4      SUCCES_COMPLET        1/7 -> 5/7            2/4   3/3   3/3     82/150        resume archive (JSON original ecrase, voir _archive_note)
ca

## 4. Verdict final, honnête

**PAS un chiffre unique -- une distribution, sur 9 lancements indépendants
connus de ce protocole exact** (même code, même graine `4242`) : 5 ont
donné `SUCCES_PARTIEL` (held-out 3/7 une fois, 4/7 quatre fois, témoin
négatif 3/3 à chaque fois), 1 a donné `SUCCES_COMPLET` (5/7), et **3 ont
divergé numériquement en NaN** (aux pas 93, 6 et 4/150 -- point d'échec non
reproductible d'un lancement à l'autre). Le taux de divergence mesuré ici
(3/9, ~33%) est plus élevé que ce qu'une première caractérisation partielle
(5 lancements, 1 divergence) avait suggéré -- corrigé ici plutôt que
reconduit tel quel, voir §3.4 pour le détail de l'investigation qui a mené à
cette révision.

**Ce qui a été vérifié sur la cause (§3.4), résumé ici** :
- Ce n'est **pas** un bug de graine manquante : `Random.seed!` + `CUDA.seed!`
  sont appelés correctement, à l'endroit correct (avant la seule source
  d'aléatoire du script), suivant la convention de ce dépôt.
- Le calcul d'entraînement lui-même est **prouvé bit-déterministe** entre
  lancements séparés, en isolation stricte, sur un protocole tronqué
  (`diag_graft_determinism_probe.jl`, 20 pas, sans phase d'éval
  avant-entraînement) : deux lancements donnent des trajectoires
  identiques à 15 chiffres significatifs.
- Mais dès que la phase d'éval avant-entraînement réelle est ajoutée
  (`diag_graft_determinism_probe_v2.jl`), la trajectoire diverge en VALEUR
  (pas en NaN) à partir du pas 4, de façon reproductible -- signe que
  l'historique d'allocation mémoire GPU (la phase d'éval, à elle seule,
  pousse la VRAM à 16041-16074/16384 Mio, sans aucun autre process actif)
  influence le chemin de calcul flottant emprunté ensuite, très
  probablement via la sélection heuristique d'algorithme de cuBLAS/CUDA
  (diagnostic plausible, non prouvé au niveau du noyau individuel).
- Sous isolation stricte et surveillée en continu (`cleanB`), le protocole
  RÉEL complet a quand même divergé (pas 4/150) -- l'isolation seule
  n'a **pas** rendu ce protocole fiable dans cet échantillon (2 relances
  propres sur 2 ont divergé).
- Une contention GPU externe RÉELLE (un autre process exécutant ce même
  notebook en parallèle) a aussi été directement constatée pendant cette
  investigation (pas seulement supposée) -- un facteur de risque
  supplémentaire confirmé, mais ni nécessaire (cleanB a divergé sans elle)
  ni suffisant à lui seul pour garantir une divergence (le lancement
  concurrent qui a contaminé `cleanA` n'a lui-même pas divergé).
- Le régime d'entraînement pré-enregistré (`LR=3e-3` fixe, sans clip de
  gradient, 150 pas sans décroissance) reste une fragilité de protocole
  plausible et non exclusive de ce qui précède : la transition vers NaN,
  quand elle survient, est un saut brutal en un seul pas depuis une perte
  encore normale (vérifié sur `cleanA`/`cleanB`), compatible aussi bien
  avec une instabilité d'optimiseur non clippé qu'avec un écart flottant
  d'origine mémoire -- non tranché avec les outils disponibles ici.

**Ce qui a marché, parmi les 6 lancements qui ont convergé** : sur Held-out
B (contrainte "oui/non", jamais vue à l'entraînement, contenu disjoint), la
conformité passe de 0/3 à 3/3 sur la majorité des lancements convergés (2/3
sur le tout premier) -- un signal de généralisation réel, présent dans
presque tous les lancements qui n'ont pas divergé. La perte d'entraînement,
quand elle ne diverge pas, baisse de façon monotone et franche (4.29 →
~0.02-0.04) sur tous les lancements convergés.

**Ce qui n'a pas marché, de façon également reproductible sur les
lancements convergés** : sur Held-out A (MÊME contrainte "un seul mot" que
l'entraînement, contenu disjoint), la conformité nette reste à 1/4 avant ET
après sur la plupart des lancements convergés (2/4 sur le lancement
`SUCCES_COMPLET` -- meilleur mais pas parfait) -- la greffe généralise donc
de façon plus stable vers une contrainte non vue (B) que vers du contenu
nouveau sous la même contrainte (A), résultat contre-intuitif mais
maintenant confirmé sur plusieurs répétitions indépendantes. Hypothèse non
vérifiée : les 6 exemples d'entraînement partagent tous le même gabarit
exact, la greffe apprend peut-être un signal plus proche de "réponds court
après CE gabarit" que "réponds court sous contrainte de format en général".

**Limite assumée et probablement pertinente** (déclarée dans le
pré-enregistrement, §7) : la perte porte sur la séquence ENTIÈRE
(prompt + réponse), pas seulement sur les tokens de réponse -- dilue le
signal de gradient. Non testée ici, laissée pour une session dédiée.

**Comparaison avec le précédent le plus proche** (goulot-vs-témoin,
`artilce/patching_cost_and_circuits.tex` §`sec:goulot`) : ce précédent a
échoué sur ses 3 critères pré-enregistrés (0 signal exploitable). Cette
expérience-ci va plus loin quand elle converge -- un signal de
généralisation réel existe sur la majorité des lancements -- mais avec un
taux de divergence numérique mesuré à ~1/3 des lancements, ce résultat est
BEAUCOUP moins fiable comme méthode reproductible qu'un chiffre unique ne
le laisserait croire. C'est la conclusion honnête de cette section : un
mécanisme qui produit un signal réel et généralisable **quand il ne
diverge pas**, sur un protocole d'entraînement dont ce dépôt a maintenant
mesuré, et non plus seulement soupçonné, la fragilité numérique.

**Mise à jour du 2026-09-01 (voir §3.5)** : un correctif ciblant la cause de
divergence tardive (décroissance de LR + arrêt anticipé déclenchés par
plateau) a été implémenté et testé sur 9 lancements indépendants
supplémentaires du protocole corrigé (6 relances dédiées + 3 exécutions
canoniques de ce notebook). Résultat mesuré : **1/9 divergence
(~11.1%)**, contre 3/9 (~33%) avant correctif -- et cette seule divergence
post-correctif est de la cause PRÉCOCE (hors périmètre de ce correctif), pas
de la cause tardive qu'il cible : **0 divergence tardive observée sur ces 9
lancements**. Sur un échantillon de cette taille, ceci n'est PAS une preuve
statistique que la cause tardive est éliminée (0 observation sur 7
lancements reste cohérent avec un taux réel non nul mais plus faible), mais
c'est la mesure honnête disponible, et elle va dans le sens attendu du
diagnostic. La conformité held-out des lancements convergés reste dans une
gamme comparable à avant (4/7 à 6/7 post-correctif, contre 3/7 à 5/7 avant)
-- le correctif n'a pas dégradé le signal de généralisation en coupant
l'entraînement trop tôt (arrêt anticipé mesuré entre les pas 79 et 90/150
selon le lancement, toujours après la fenêtre de convergence réelle). La
cause précoce (§3.4 point 3, allocation mémoire GPU) reste, elle, **non
résolue** -- une piste de préchauffage GPU a été testée (§3.5) et n'a
apporté qu'une amélioration partielle (retarde la divergence de 2 pas sans
l'éliminer, une nouvelle discontinuité apparaissant au pas 6 au lieu du pas
4).


## 5. Ce que ce notebook démontre, et ce qu'il ne démontre pas

**Démontré** :
- Le mécanisme greffe + gel + backward élagué fonctionne correctement à
  l'échelle réelle de Qwen2.5-1.5B-Instruct (poids HuggingFace réels, jamais
  entraînés par NeuroDSL) -- gel à 100%, bit-exactness du gate à zéro,
  échappement du gate après un pas, ~31x d'accélération du backward élagué.
- Qwen2.5-1.5B-Instruct a un échec réel et reproductible de suivi
  d'instruction sous distraction (contrainte de format ignorée en présence
  d'une clause d'explication).
- Une greffe minimale (10 tenseurs, backbone entièrement gelé) entraînée sur
  6 exemples produit, **quand l'entraînement ne diverge pas** (6 lancements
  sur 9), un signal de correction qui généralise partiellement -- 3/7 à
  5/7 held-out selon le lancement, 2/3 ou 3/3 sur une contrainte jamais vue
  -- sans effondrement dégénéré (3/3 témoin négatif préservé à chaque fois).
- **Ce protocole d'entraînement exact (LR=3e-3 fixe, 150 pas, sans clip de
  gradient ni décroissance) est numériquement fragile** -- mesuré ici à
  **3 divergences sur 9 lancements indépendants (~33%)**, à des pas
  différents et imprévisibles (93, 6, 4). Ce n'est pas un vernis
  méthodologique : une investigation dédiée (§3.4) a établi que (a) ce
  n'est pas un bug de graine (seedage vérifié correct), (b) le calcul est
  bit-déterministe en isolation stricte sur un protocole tronqué, (c) la
  phase d'évaluation avant-entraînement, à elle seule, pousse la VRAM à
  saturation quasi complète de la carte 16 Go et modifie mesurablement le
  chemin de calcul flottant qui suit (probablement via la sélection
  d'algorithme cuBLAS dépendante de l'état mémoire), (d) l'isolation
  stricte et surveillée en continu (`cleanB`) n'a **pas** empêché une
  divergence, et (e) une contention GPU externe réelle a été constatée
  directement pendant cette même investigation (facteur de risque
  supplémentaire confirmé, mais ni nécessaire ni suffisant à lui seul).
  Rapporté ici en détail précisément parce qu'il aurait été facile de ne
  rapporter que le premier lancement (réussi), ou de s'arrêter à une
  explication simple et rassurante ("juste de la contention", ou "juste un
  bug de graine") qui n'a pas résisté à la vérification.

**PAS démontré** : que ce mécanisme "répare" le suivi d'instruction de
façon fiable ou générale ; qu'un budget d'entraînement plus grand,
un masquage de perte sur la réponse, un clip de gradient, un autre site de
greffe, ou une autre graine donneraient un résultat meilleur (ou pire, ou
plus stable) -- aucun balayage d'hyperparamètre n'a été fait au-delà de la
caractérisation de reproductibilité du §3.4 ; que ce résultat se transfère
à d'autres familles d'échecs (comptage de lettres notamment, écarté
précisément parce que jugé peu plausible pour ce mécanisme) ; **quel est,
au niveau du noyau CUDA individuel, le mécanisme exact qui rend le calcul
sensible à l'historique d'allocation mémoire** -- diagnostiqué comme
plausible (sélection d'algorithme cuBLAS) mais pas instrumenté à ce niveau
faute de temps ; **si un clip de gradient ou une réduction de la VRAM de
pointe (ex. libérer la mémoire entre l'éval et l'entraînement) éliminerait
la divergence** -- non testé, aurait nécessité de modifier le protocole
pré-enregistré, ce qui n'a pas été fait ici.

**Note sur le critère de succès/partiel/échec pré-enregistré** (§6 du
pré-enregistrement) : ce critère ne mentionnait ni ne présupposait
explicitement une exigence de stabilité d'un lancement à l'autre -- il a
été écrit et fixé pour UN lancement, avant qu'aucune divergence ne soit
connue. Le fait qu'un protocole numériquement instable ait pu passer un
pré-enregistrement sans qu'un critère de stabilité soit prévu est lui-même
signalé ici comme une limite du pré-enregistrement original, pas quelque
chose à passer sous silence -- voir l'addendum daté dans
`graft_qwen_experiment_preregistration.md` §8.

**Artefacts produits par ce notebook et par cette investigation**, tous
sous `notebook/` : `graft_qwen_experiment_preregistration.md`
(pré-enregistrement complet + addendum daté), `graft_qwen_failure_search.jl`/`.log`
(recherche de l'échec), `graft_qwen_experiment_run.jl` (script principal,
avec la garde de non-finitude), `graft_qwen_experiment_run{,1,2,3}.log` +
`graft_qwen_experiment_results{,_run1,_run2,_run3}.json` (lancements
historiques), `graft_qwen_experiment_cleanA.log`/`cleanB.log` +
`_results_cleanA.json`/`_results_cleanB.json` (relances isolées de cette
session, toutes deux divergentes), `graft_qwen_experiment_results_ipynb2.json`
(résultat du lancement `nbconvert` concurrent découvert pendant cette
investigation), `diag_graft_determinism_probe.jl`/`_v2.jl` +
`_A.log`/`_B.log`/`_v2.log` (sondes de déterminisme isolées, §3.4), et
`graft_qwen_check{1,2a,2b,3}_*.jl` + leurs `.log`/`.json` (vérifications de
correction, écrites avant cette expérience).

**Mise à jour du 2026-09-01** : la fragilité numérique documentée ci-dessus
a reçu un correctif ciblé (décroissance de LR + arrêt anticipé déclenchés
par plateau, voir §3.5) qui NE change PAS le protocole pré-enregistré (LR
initial, budget de pas, données). Mesuré sur 9 lancements supplémentaires
(6 relances dédiées + 3 exécutions canoniques de ce notebook) :
taux de divergence 1/9 (~11.1%) contre 3/9 (~33%) avant, et la seule
divergence restante est de la cause précoce (allocation mémoire GPU) que ce
correctif ne cible pas -- 0 divergence tardive observée sur ces 9
lancements. Échantillon trop petit pour un résultat statistiquement
tranché, rapporté tel quel plutôt que présenté comme « résolu ». Une piste
de préchauffage GPU pour la cause précoce a aussi été testée et n'a apporté
qu'une aide partielle (retarde le point de divergence de 2 pas sans
l'éliminer) -- cette cause reste ouverte. Artefacts supplémentaires produits
par cette phase, tous sous `notebook/` : `graft_qwen_experiment_run_fix{1-6}.log`
+ `graft_qwen_experiment_results_fix{1-6}.json` (les 6 relances dédiées de
re-vérification), `diag_graft_determinism_probe_v3_warmup.jl`/`.log` (piste
de préchauffage pour la cause précoce).
